# 🧠 NeuralAI Mamba K2 Training
## First Owned Base Model — 790M Parameters

**Base**: `state-spaces/mamba-790m-hf` (SSM architecture)
**Scale**: 10,000+ instruction samples, 500–1000 SFT steps, optional DPO
**Output**: Merged model → HuggingFace `Subject-Emu-5259/NeuralAI-Mamba-K2`
**GGUF**: Q4_K_M quant for LM Studio

---
## Cell 1: Install Dependencies

In [ ]:
!pip install -q torch transformers datasets peft accelerate trl huggingface_hub mamba-ssm

## Cell 2: Import & Config

In [ ]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from datasets import load_dataset, concatenate_datasets
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, DPOTrainer, DPOConfig
from huggingface_hub import HfApi, create_repo
import json

# === CONFIG ===
BASE_MODEL = "state-spaces/mamba-790m-hf"
OUTPUT_DIR = "./mamba-k2-sft"
DPO_OUTPUT_DIR = "./mamba-k2-dpo"
MERGED_DIR = "./mamba-k2-merged"
HF_REPO = "Subject-Emu-5259/NeuralAI-Mamba-K2"

# SFT settings
SFT_SAMPLES = 10_000  # target instruction samples
SFT_EPOCHS = 3
SFT_BATCH_SIZE = 4
SFT_GRAD_ACCUM = 8
SFT_LR = 2e-4
MAX_SEQ_LENGTH = 2048

# LoRA settings
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "N/A")

## Cell 3: Load 10K+ Instruction Data
Combines UltraChat, OpenHermes, and custom chat data for diverse coverage.

In [ ]:
def load_instruction_data():
    """Load 10K+ instruction-following samples from multiple sources."""
    datasets_list = []
    
    # UltraChat 200k — high-quality multi-turn conversations
    print("Loading UltraChat...")
    try:
        ultra = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:5000]")
        ultra = ultra.map(lambda x: {"text": format_chatml(x["messages"])})
        datasets_list.append(ultra.select_columns(["text"]))
        print(f"  UltraChat: {len(ultra)} samples")
    except Exception as e:
        print(f"  UltraChat failed: {e}")
    
    # OpenHermes — diverse instruction data
    print("Loading OpenHermes...")
    try:
        hermes = load_dataset("teknium/OpenHermes-2.5", split="train[:3000]")
        hermes = hermes.map(lambda x: {"text": format_single_turn(x["conversations"])})
        datasets_list.append(hermes.select_columns(["text"]))
        print(f"  OpenHermes: {len(hermes)} samples")
    except Exception as e:
        print(f"  OpenHermes failed: {e}")
    
    # Code instructions — code generation capability
    print("Loading Code-Feedback...")
    try:
        code = load_dataset("m-a-p/Code-Feedback", split="train[:2000]")
        code = code.map(lambda x: {"text": format_code_instruction(x)})
        datasets_list.append(code.select_columns(["text"]))
        print(f"  Code: {len(code)} samples")
    except Exception as e:
        print(f"  Code failed: {e}")
    
    # Orca reasoning data
    print("Loading Orca reasoning...")
    try:
        orca = load_dataset("microsoft/orca-math-word-problems-200k", split="train[:1000]")
        orca = orca.map(lambda x: {"text": f"<|im_start|>user\nSolve: {x['question']}<|im_end|>\n<|im_start|>assistant\n{x['answer']}<|im_end|>"})
        datasets_list.append(orca.select_columns(["text"]))
        print(f"  Orca: {len(orca)} samples")
    except Exception as e:
        print(f"  Orca failed: {e}")
    
    combined = concatenate_datasets(datasets_list)
    combined = combined.shuffle(seed=42)
    print(f"\nTotal: {len(combined)} instruction samples")
    return combined

def format_chatml(messages):
    """Format list of {role, content} into ChatML string."""
    parts = []
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
    return "\n".join(parts)

def format_single_turn(conversations):
    """Format a single-turn conversation."""
    if isinstance(conversations, list):
        return format_chatml(conversations)
    return conversations

def format_code_instruction(x):
    """Format code instruction-style data."""
    prompt = x.get("instruction", x.get("prompt", x.get("question", "")))
    answer = x.get("output", x.get("response", x.get("answer", "")))
    return f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{answer}<|im_end|>"

dataset = load_instruction_data()
print(f"\nFirst sample preview:\n{dataset[0]['text'][:500]}...")

## Cell 4: Load Base Model + Tokenizer

In [ ]:
print(f"Loading {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

# GPTNeoX tokenizer: use eos_token as pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} params")

# Apply LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["x_proj", "in_proj", "out_proj"],  # Mamba-specific projection layers
    bias="none"
)
model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA: {trainable:,} trainable / {total:,} total ({100*trainable/total:.1f}%)")

## Cell 5: SFT Training (500–1000 steps)

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=SFT_BATCH_SIZE,
    gradient_accumulation_steps=SFT_GRAD_ACCUM,
    learning_rate=SFT_LR,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    save_total_limit=3,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=False,  # Mamba doesn't support gradient checkpointing well
    optim="adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,  # pack multiple samples into each sequence
    dataset_text_field="text",
)

print(f"Starting SFT: {len(dataset)} samples × {SFT_EPOCHS} epochs = ~{len(dataset)*SFT_EPOCHS//(SFT_BATCH_SIZE*SFT_GRAD_ACCUM)} steps")
trainer.train()

# Save LoRA adapter
trainer.save_model(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
print(f"SFT complete. Adapter saved to {OUTPUT_DIR}/final_adapter")

## Cell 6: Merge LoRA & Save Full Model

In [ ]:
print("Merging LoRA adapter into base model...")
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

# Save chat template
chat_template = """{% for message in messages %}
{% if message['role'] == 'system' %}
<|im_start|>system
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'user' %}
<|im_start|>user
{{ message['content'] }}<|im_end|>
{% elif message['role'] == 'assistant' %}
<|im_start|>assistant
{{ message['content'] }}<|im_end|>
{% endif %}
{% endfor %}
{% if add_generation_prompt %}
<|im_start|>assistant
{% endif %}"""

with open(f"{MERGED_DIR}/chat_template.jinja", "w") as f:
    f.write(chat_template)

print(f"Merged model saved to {MERGED_DIR}")
print(f"Size: {sum(os.path.getsize(os.path.join(root, f)) for root, _, files in os.walk(MERGED_DIR) for f in files) / 1e9:.2f} GB")

## Cell 7: Quick Generation Test

In [ ]:
def test_generation(prompt, max_new=100):
    messages = [
        {"role": "system", "content": "You are NeuralAI Mamba K2, a helpful AI assistant. Be accurate, concise, and thoughtful."},
        {"role": "user", "content": prompt}
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1
    )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

tests = [
    "What is the capital of France?",
    "Write a Python function to compute Fibonacci numbers.",
    "Explain the concept of neural networks in simple terms."
]

for t in tests:
    print(f"\n{'='*60}")
    print(f"USER: {t}")
    response = test_generation(t)
    print(f"K2: {response}")
    print(f"{'='*60}")

## Cell 8: Push to HuggingFace

In [ ]:
from huggingface_hub import HfApi, create_repo
from google.colab import userdata

# Load HF token from Colab secrets
HF_TOKEN = userdata.get('HF_TOKEN')

# Create repo if needed
api = HfApi()
try:
    create_repo(HF_REPO, token=HF_TOKEN, private=False, exist_ok=True)
    print(f"Repo {HF_REPO} ready")
except Exception as e:
    print(f"Repo creation note: {e}")

# Upload merged model
api.upload_folder(
    folder_path=MERGED_DIR,
    repo_id=HF_REPO,
    repo_type="model",
    token=HF_TOKEN,
    commit_message="Mamba K2: 790M SFT on 10K+ instruction samples | NeuralAI first owned base"
)
print(f"Uploaded to https://huggingface.co/{HF_REPO}")

## Cell 9 (Optional): GGUF Conversion
Converts to Q4_K_M for LM Studio / llama.cpp. Requires ~3GB for conversion.

In [ ]:
# This cell converts the merged model to GGUF Q4_K_M
# You can skip and use a pre-converted GGUF: mradermacher/mamba-790m-hf-GGUF

GGUF_RUN = False  # Set to True to convert

if GGUF_RUN:
    !pip install -q gguf
    
    # Convert to FP16 first, then quantize
    !python llm/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
        --outtype f16 \
        --outfile {MERGED_DIR}/NeuralAI-Mamba-K2-F16.gguf
    
    # Quantize to Q4_K_M
    !llm/llama.cpp/quantize {MERGED_DIR}/NeuralAI-Mamba-K2-F16.gguf \
        {MERGED_DIR}/NeuralAI-Mamba-K2-Q4_K_M.gguf Q4_K_M
    
    !ls -lh {MERGED_DIR}/*.gguf
    print("GGUF conversion complete!")
else:
    print("GGUF conversion skipped. Use pre-converted mradermacher/mamba-790m-hf-GGUF instead.")

---
## Done! Next Steps

1. **Register in model_manager.py**: Add `neuralai-mamba-k2` entry
2. **Run benchmark evals**: Use `benchmark_k2.py`
3. **Deploy GGUF**: Download Q4_K_M to `models/` and restart llmster
4. **Web UI integration**: Update model selector and system prompt